# Spam II: Naive Bayes on real SMS: hard frequencies and numerical stability

In the previous notebook a word counted once per message and we multiplied a handful of probabilities. Real messages are
longer and the vocabulary has thousands of words. In this notebook we

1. load the **SMS Spam Collection** properly (duplicates, stratified split);
2. move from *presence* to **hard frequencies** (word counts, the *multinomial* model);
3. see the model **break** on long messages, and fix it with **log-space arithmetic** and the *log-sum-exp* trick;
4. evaluate with precision / recall (not accuracy) and choose an operating point;
5. read what the model learned: the ranked word list is the **attacker's roadmap**.

In [ ]:
import numpy as np
import scipy.sparse as sp
from matplotlib import pyplot as plt
from scipy.special import logsumexp
from sklearn.metrics import ConfusionMatrixDisplay, precision_recall_curve
from sklearn.naive_bayes import MultinomialNB

import spamlib as sl

plt.rcParams["figure.dpi"] = 100

## 1. Data

5,574 SMS messages (Almeida et al., 2011), of which ~13% are spam. Exact duplicates are removed *before* splitting: if the same
spam text sits in both train and test, the test score measures memorisation, not generalisation.

In [ ]:
raw_texts, _ = sl.load(dedupe=False)
texts, y = sl.load()
print(f"messages: {len(raw_texts)} -> {len(texts)} after removing duplicates")
print(f"spam ratio: {y.mean():.3f}")

X_txt_train, X_txt_test, y_train, y_test = sl.split(texts, y)
print(f"train {len(X_txt_train)}  test {len(X_txt_test)}  (spam in test: {y_test.sum()})")

## 2. From text to a count matrix (bag of words)

The **bag of words** keeps *how many times* each word occurs and forgets the order. The tokenizer lower-cases, keeps
`!`, `£`, `$` (strong spam cues) and maps long digit runs (phone numbers, short codes) to one token, `num`.

In [ ]:
print(sl.tokenize("URGENT! You won a £1000 prize. Call 09061701461 now!!"))

In [ ]:
def build_vocab(docs, min_df=2):
    """Words that occur in at least `min_df` training messages, in a fixed order."""
    df = {}
    for d in docs:
        for w in set(sl.tokenize(d)):
            df[w] = df.get(w, 0) + 1
    return sorted(w for w, c in df.items() if c >= min_df)


def count_matrix(docs, index):
    """Sparse (n_docs x |V|) matrix of raw word counts. Out-of-vocabulary words are ignored."""
    rows, cols, vals = [], [], []
    for i, d in enumerate(docs):
        counts = {}
        for w in sl.tokenize(d):
            j = index.get(w)
            if j is not None:
                counts[j] = counts.get(j, 0) + 1
        rows += [i] * len(counts)
        cols += counts.keys()
        vals += counts.values()
    return sp.csr_matrix((vals, (rows, cols)), shape=(len(docs), len(index)), dtype=np.float64)


vocab = build_vocab(X_txt_train)
index = {w: i for i, w in enumerate(vocab)}
X_train = count_matrix(X_txt_train, index)
X_test = count_matrix(X_txt_test, index)
print(f"vocabulary: {len(vocab)} words; X_train {X_train.shape}, density {X_train.nnz / np.prod(X_train.shape):.4f}")

## 3. Multinomial Naive Bayes with hard frequencies

A message is now a *bag* of $n$ word tokens drawn from class-specific word distributions:

$$
P(w\mid y)=\frac{N_{w,y}+k}{\sum_{w'}N_{w',y}+k\,|V|}\qquad
\log P(y\mid x)\;\propto\;\log P(y)+\sum_{w}x_w\,\log P(w\mid y)
$$

where $N_{w,y}$ is the number of times word $w$ occurs in messages of class $y$ and $x_w$ the count in the message.
The score of a message is a **dot product** between its count vector and a table of log-probabilities.

In [ ]:
class NaiveBayes:
    """Multinomial Naive Bayes, everything in log space."""

    def __init__(self, k=1.0, binary=False):
        self.k = k
        self.binary = binary  # binary=True -> presence instead of counts

    def _prep(self, X):
        return (X > 0).astype(np.float64) if self.binary else X

    def fit(self, X, y):
        X = self._prep(X)
        self.log_prior_ = np.log(np.bincount(y) / len(y))
        n = np.vstack([np.asarray(X[y == c].sum(axis=0)).ravel() for c in (0, 1)])  # N_{w,y}, shape (2, |V|)
        self.log_lik_ = np.log(n + self.k) - np.log(n.sum(axis=1, keepdims=True) + self.k * n.shape[1])
        return self

    def joint_log_proba(self, X):
        """log P(y) + sum_w x_w log P(w|y), shape (n_docs, 2)."""
        return self._prep(X) @ self.log_lik_.T + self.log_prior_

    def predict_log_proba(self, X):
        jll = self.joint_log_proba(X)
        return jll - logsumexp(jll, axis=1, keepdims=True)  # normalise in log space

    def predict_proba(self, X):
        return np.exp(self.predict_log_proba(X))

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X)[:, 1] >= threshold).astype(int)


nb = NaiveBayes(k=1.0).fit(X_train, y_train)
pred = nb.predict(X_test)
sl.show({"NB counts (k=1)": sl.scores(y_test, pred, nb.predict_proba(X_test)[:, 1])})

### Sanity check against scikit-learn

A from-scratch implementation earns trust when it reproduces a reference implementation.

In [ ]:
ref = MultinomialNB(alpha=1.0).fit(X_train, y_train)
print("max |P_ours - P_sklearn| =", np.abs(nb.predict_proba(X_test) - ref.predict_proba(X_test)).max())

## 4. Why logarithms? Underflow

A 100-word message multiplies 100 probabilities of order $10^{-3}$: the product is $10^{-300}$, at the edge of what a
64-bit float can represent (the smallest positive `float64` is about $10^{-308}$). Slightly longer, and the result is exactly
**0.0**. Then $P(\text{spam}\mid x)=\frac{0}{0+0}$ is **NaN**.

In [ ]:
long_msg = " ".join(t for t, lab in zip(X_txt_train, y_train) if lab == 1)[:6000]
x_long = count_matrix([long_msg], index)
print(f"tokens in the message: {int(x_long.sum())}")

lik = np.exp(nb.log_lik_)  # P(w|y) as ordinary probabilities, shape (2, |V|)
prior = np.exp(nb.log_prior_)
prod = prior * np.prod(lik ** np.asarray(x_long.todense()), axis=1)  # naive product of probabilities
print("naive product      :", prod)
with np.errstate(invalid="ignore"):
    print("naive posterior    :", prod / prod.sum())

print("log-space posterior:", nb.predict_proba(x_long))
print("joint log-prob     :", nb.joint_log_proba(x_long))

### The log-sum-exp trick

Working with $\ell_y=\log P(y)+\sum_w x_w\log P(w\mid y)$, the posterior is a **softmax** of the two scores. Subtract the
maximum before exponentiating so that the largest term is $e^0=1$:

$$
\log\sum_y e^{\ell_y}= m+\log\sum_y e^{\ell_y-m},\qquad m=\max_y \ell_y
$$

Rules of thumb: (i) *multiply* probabilities $\to$ *add* log-probabilities; (ii) never exponentiate a very negative number
on its own; (iii) `scipy.special.logsumexp` does the max-shift for you.

In [ ]:
ell = nb.joint_log_proba(x_long)[0]
m = ell.max()
print("manual  :", np.exp(ell - m) / np.exp(ell - m).sum())
print("logsumexp:", np.exp(ell - logsumexp(ell)))

## 5. Discrete (presence) vs. hard frequencies (counts), and the smoothing constant $k$

In [ ]:
results = {}
for binary in (True, False):
    for k in (0.1, 1.0):
        m_ = NaiveBayes(k=k, binary=binary).fit(X_train, y_train)
        name = f"NB {'presence' if binary else 'counts'} k={k}"
        results[name] = sl.scores(y_test, m_.predict(X_test), m_.predict_proba(X_test)[:, 1])
sl.show(results)

## 6. Evaluate like a defender: confusion matrix and operating point

**Accuracy hides the problem.** With 87% ham, a filter that never flags anything scores 87%. What matters:

* **precision** $=TP/(TP+FP)$: of the messages we block, how many were really spam (false positives block a real message);
* **recall** $=TP/(TP+FN)$: of the spam that exists, how much do we catch (false negatives reach the inbox).

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["ham", "spam"], colorbar=False)
plt.show()

NB posteriors are extreme (almost always near 0 or 1) because the independence assumption counts correlated words twice.
The *ranking* is useful, the absolute probability is not calibrated. We pick a threshold from the precision-recall curve:
a mail provider would rather let some spam through than block a legitimate message.

In [ ]:
proba = nb.predict_proba(X_test)[:, 1]
prec, rec, thr = precision_recall_curve(y_test, proba)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(rec, prec)
ax[0].set(xlabel="recall", ylabel="precision", title="Precision-recall curve")
ax[1].plot(thr, prec[:-1], label="precision")
ax[1].plot(thr, rec[:-1], label="recall")
ax[1].set(xlabel="decision threshold on P(spam|x)", title="Operating point")
ax[1].legend()
plt.tight_layout()
plt.show()

ok = np.where(prec[:-1] >= 0.99)[0]
if len(ok):
    i = ok[0]
    print(f"threshold {thr[i]:.4f}: precision {prec[i]:.3f}, recall {rec[i]:.3f}")

## 7. What did the model learn? (the attacker's roadmap)

The log-odds $\log\frac{P(w\mid\text{spam})}{P(w\mid\text{ham})}$ is each word's *vote*. To an attacker who can read (or
estimate) this table, the two ends of the list are a recipe:

* words with **high** log-odds must be **removed** or disguised (`free`, `claim`, `£`...);
* words with **very negative** log-odds are **"good words"** to append (`sorry`, `later`, `lor`...): each one shifts the score
  towards ham. This is the *good-word attack* of Lowd & Meek (2005); we implement it in notebook 07.

In [ ]:
spam_w, ham_w = sl.top_log_odds(vocab, nb.log_lik_[1], nb.log_lik_[0], n=15)
print("most spammy :", ", ".join(f"{w} ({v:.1f})" for w, v in spam_w))
print("most hammy  :", ", ".join(f"{w} ({v:.1f})" for w, v in ham_w))

Look at `lt` and `gt`: they are the leftovers of the HTML escapes `&lt;#&gt;` that appear in the *ham* half of the corpus,
not real evidence of legitimacy. This is a **spurious correlation** (a data-collection artefact): the model learned where the
benign messages came from, not what makes them benign (Arp et al., 2022).

## Exercises

1. Plot precision and recall as a function of the vocabulary size (`min_df` = 1, 2, 5, 10, 20). Which side degrades first?
2. Replace `nb.predict_proba` by the naive product on the whole test set. For how many messages is the result `NaN`?
3. `k` acts as a prior on unseen words. Explain why `k=0.1` gives *more extreme* posteriors than `k=1`.
4. Repeat the experiment without removing duplicates. By how much do the scores change, and why is that misleading?